In [1]:
import json

def get_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [ ]:
chunk_store = get_jsonl_data('data/chunk_store.jsonl')
print(chunk_store[0])

corpus = get_jsonl_data('data/corpus.jsonl')
print(corpus[0])

{'doc_id': 'doc_0', 'article_id': 'article_0', 'chunk_id': 0, 'text': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản'}
{'doc_id': 'doc_0', 'article_id': 'article_0', 'article_content': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.\n\n2. Tranh chấp quốc tế về địa

In [3]:
reader_data = []

# Build a fast lookup: article_id -> list of chunk texts
chunks_by_article = {}
for chunk in chunk_store:
    aid = chunk["article_id"]
    chunks_by_article.setdefault(aid, []).append(chunk["text"] )

for row in corpus:
    article_id = row["article_id"]
    context_chunks = chunks_by_article.get(article_id, [])

    qa_pairs = row.get("generated_qa_pairs", [])
    for qa in qa_pairs:
        question = qa.get("question", "").strip()
        answer = qa.get("answer", "").strip()

        # Keep only usable samples.
        if not question or not answer or not context_chunks:
            continue

        reader_data.append({
            "question": question,
            "context": context_chunks,
            "answer": answer,
            "article_id": article_id
        })

print(f"Total reader samples: {len(reader_data)}")
print("First sample:", reader_data[0] if reader_data else "No valid sample")

# Quick sanity checks
empty_context = sum(1 for x in reader_data if not x["context"])
print(f"Samples with empty context: {empty_context}")

Total reader samples: 29145
First sample: {'question': 'Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?', 'context': ['Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản', '1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam;', 'bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.', '2. Tranh chấp quốc tế v

In [ ]:
from pathlib import Path
import json

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

reader_data_jsonl_path = output_dir / "reader_data.jsonl"

# Save one JSON record per line for downstream training.
with reader_data_jsonl_path.open("w", encoding="utf-8") as f:
    for row in reader_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(reader_data)} sample(s) to {reader_data_jsonl_path}")

Saved 29145 sample(s) to processed\reader_data.jsonl
